In [1]:
import torch
from sklearn.model_selection import train_test_split
import pandas as pd
import torch.nn as nn
import numpy as np
from sklearn.preprocessing import StandardScaler
import csv

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
meteo_old = pd.read_csv("/./in-sample/2024_meteo_act_comuneimpianto.csv", sep=";")
meteo= pd.read_csv("./in-sample/augmented_data_finale.csv", sep=",")
prod = pd.read_excel("./in-sample/produzione_2024.xlsx")

In [4]:
meteo.head()

,Dataora,wds_comuneimpianto,wdd_comuneimpianto,rnf_comuneimpianto,rhm_comuneimpianto,wds_comunelimitrofo1,wdd_comunelimitrofo1,rnf_comunelimitrofo1,rhm_comunelimitrofo1,wds_comunelimitrofo2,...,media_wds3_tutti,std_wds3_tutti,media_u_tutti,std_u_tutti,media_v_tutti,std_v_tutti,media_rnf_tutti,std_rnf_tutti,media_rhm_tutti,std_rhm_tutti
0,2024-01-01 00:00:00,16.8,204.3,0.0,82.6,22.0,223.3,0.0,84.5,22.1,...,9169.419400,2412.420042,-15.435112,0.669001,-13.584405,3.737179,0.0,0.0,87.08,3.526613
1,2024-01-01 01:00:00,16.5,200.6,0.0,82.5,24.1,222.2,0.0,84.6,24.2,...,10867.078867,3486.115394,-16.719496,1.147870,-14.446141,4.844506,0.0,0.0,86.94,3.434094
2,2024-01-01 02:00:00,15.2,199.0,0.0,82.7,25.8,223.7,0.0,84.2,25.9,...,12355.717222,4674.618379,-17.348976,1.847225,-15.276205,5.781740,0.0,0.0,86.60,3.261135
3,2024-01-01 03:00:00,13.3,202.8,0.0,83.4,24.4,224.6,0.0,84.2,25.1,...,12258.144844,4884.092865,-16.196080,2.317481,-15.058037,5.557419,0.0,0.0,87.40,3.595136
4,2024-01-01 04:00:00,11.7,198.6,0.0,84.5,21.5,226.7,0.0,84.7,24.1,...,11537.142151,5038.222146,-14.979914,2.393150,-14.468829,6.098506,0.0,0.0,87.92,3.262208


In [5]:
#meteo["month"] = pd.to_datetime(meteo["Dataora"]).dt.month
#meteo["month"] = np.sin(meteo["month"]*2*3.14/12)
target = "Mwh"
Y = prod[target]
X = meteo.drop(columns=["Dataora"])
unito = pd.concat([X, Y], axis=1)

In [6]:
unito.dropna(inplace=True)

In [7]:
Y = unito[target]
X = unito.drop(columns=[target])
X.head()

print(X.shape)

(8733, 96)


In [8]:
train_ratio = 0.80
validation_ratio = 0.20
x_train, x_valid, y_train, y_valid = train_test_split(X, Y, test_size=1 - train_ratio, shuffle=False)
x_train2 = X.iloc[:7000,:4]
x_valid2 = X.iloc[7000:,:4]
y_train2 = Y.iloc[:7000]
y_valid2 = Y.iloc[7000:]
x_train2.shape

(7000, 4)

In [9]:
x_train = torch.from_numpy(x_train.values).float()
y_train = torch.from_numpy(y_train.values).float()
x_valid = torch.from_numpy(x_valid.values).float()
y_valid = torch.from_numpy(y_valid.values).float()

x_train2 = torch.from_numpy(x_train2.values).float()
y_train2 = torch.from_numpy(y_train2.values).float()
x_valid2 = torch.from_numpy(x_valid2.values).float()
y_valid2 = torch.from_numpy(y_valid2.values).float()


In [10]:
class mydataset:
    def __init__(self, x, y):
        self.x = x
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.x[idx,:], self.y[idx]

In [11]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.LayerNorm(96),
            nn.Linear(96, 64),
            nn.ReLU(),
            nn.LayerNorm(64),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.LayerNorm(32),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.layers(x)

In [4]:
class MLP2(nn.Module):
    def __init__(self):
        super().__init__()
        self.layers = nn.Sequential(
            nn.LayerNorm(96),
            nn.Linear(96, 32),
            nn.ReLU(),
            nn.LayerNorm(32),
            nn.Linear(32, 1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.layers(x)

In [37]:
net = MLP2().to(device)
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(net.parameters(), lr=0.001,  weight_decay=1e-4)


In [32]:
traindata = mydataset(x_train, y_train)
validdata = mydataset(x_valid, y_valid)

trainloader = torch.utils.data.DataLoader(traindata, batch_size=16, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(validdata, batch_size=16, shuffle=True, num_workers=2)

alldata = torch.utils.data.ConcatDataset([traindata, validdata])
allloader = torch.utils.data.DataLoader(alldata, batch_size=16, shuffle=True, num_workers=2)

In [33]:
def evalstep(epoch):

    net.eval()
    outputs = []
    with torch.no_grad():
        for i in range(len(validdata)):
            inputs, targets = validdata.x[i,:], validdata.y[i]
            if inputs.dim() == 1:
                inputs = inputs.unsqueeze(0)
            inputs, targets = inputs.float().to(device), targets.float().to(device)

            outputs.append(net(inputs))

        
    loss = custom_loss(outputs, validdata.y)
    print(f'Epoch: {epoch} Custom Loss: {loss.item()}')

In [34]:
def train_step(model, optimizer, loss_function, trainloader, epochs = 20):
    for epoch in range(0, epochs):
        print(f'Starting Epoch {epoch+1}')

        current_loss = 0.0

        for i, data in enumerate(trainloader, 0):
            inputs, targets = data
            inputs, targets = inputs.float().to(device), targets.float().to(device)
     
            targets = targets.reshape((targets.shape[0], 1))

            optimizer.zero_grad()

            outputs = model(inputs)

            loss = loss_function(outputs, targets)

            loss.backward()

            optimizer.step()

            current_loss += loss.item()
        evalstep(epoch)
        print(f'Epoch {epoch+1} finished')

    print("Training has completed")

In [35]:
def custom_loss(outputs, targets):
    sum = 0
    for i in range(len(outputs)):
        sum += abs(outputs[i] - targets[i])
    return sum / abs(targets).sum()

In [ ]:
train_step(net, optimizer, loss_function, trainloader, epochs=300)

In [11]:
def evalstep(epoch):

    net.eval()
    outputs = []
    with torch.no_grad():
        for i in range(len(validdata)):
            inputs, targets = validdata.x[i,:], validdata.y[i]
            if inputs.dim() == 1:
                inputs = inputs.unsqueeze(0)
            inputs, targets = inputs.float().to(device), targets.float().to(device)

            outputs.append(net(inputs))

        
    loss = custom_loss(outputs, validdata.y)
    print(f'Epoch: {epoch} Custom Loss: {loss.item()}')

In [38]:
train_step(net, optimizer, loss_function, allloader, epochs=35)

Starting Epoch 1
Epoch: 0 Custom Loss: 0.5213763117790222
Epoch 1 finished
Starting Epoch 2
Epoch: 1 Custom Loss: 0.48138076066970825
Epoch 2 finished
Starting Epoch 3
Epoch: 2 Custom Loss: 0.4605843126773834
Epoch 3 finished
Starting Epoch 4
Epoch: 3 Custom Loss: 0.44761210680007935
Epoch 4 finished
Starting Epoch 5
Epoch: 4 Custom Loss: 0.4676322638988495
Epoch 5 finished
Starting Epoch 6
Epoch: 5 Custom Loss: 0.5413516759872437
Epoch 6 finished
Starting Epoch 7
Epoch: 6 Custom Loss: 0.44937267899513245
Epoch 7 finished
Starting Epoch 8
Epoch: 7 Custom Loss: 0.39490774273872375
Epoch 8 finished
Starting Epoch 9
Epoch: 8 Custom Loss: 0.4058789610862732
Epoch 9 finished
Starting Epoch 10
Epoch: 9 Custom Loss: 0.41580653190612793
Epoch 10 finished
Starting Epoch 11
Epoch: 10 Custom Loss: 0.43149566650390625
Epoch 11 finished
Starting Epoch 12
Epoch: 11 Custom Loss: 0.3926028311252594
Epoch 12 finished
Starting Epoch 13
Epoch: 12 Custom Loss: 0.36667588353157043
Epoch 13 finished
Startin

In [39]:
torch.save(net.state_dict(), "eznnfinal.pth")

In [16]:
class LSTMModel(nn.Module):
    def __init__(self, input_dim, hidden_dim, layer_dim, output_dim):
        super(LSTMModel, self).__init__()
        self.hidden_dim = hidden_dim
        self.layer_dim = layer_dim
        self.norm = nn.LayerNorm(input_dim)
        self.lstm = nn.LSTM(input_dim, hidden_dim, layer_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x, h0=None, c0=None):
        if h0 is None or c0 is None:
            h0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
            c0 = torch.zeros(self.layer_dim, x.size(0), self.hidden_dim).to(x.device)
        x = self.norm(x)
        out, (hn, cn) = self.lstm(x, (h0, c0))
        out = self.fc(out[:, -1, :])
        return out, hn, cn

In [38]:
model = LSTMModel(input_dim=20, hidden_dim=64, layer_dim=1, output_dim=1)
model.to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.1, weight_decay=1e-5)

In [39]:
sequence_length = 5

def create_sequences(x, y, seq_len):
    xs, ys = [], []
    for i in range(len(x) - seq_len):
        xs.append(x[i:i+seq_len])
        ys.append(y[i+seq_len])
    return torch.stack(xs), torch.stack(ys)

x_seq, y_seq = create_sequences(x_train2, y_train2, sequence_length)


In [40]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=1e-4)

In [41]:
def predict_on_x_valid(model, x_valid, device='cuda'):
    model.eval()
    
    x_valid_seq = x_valid.unsqueeze(1).to(device)

    with torch.no_grad():
        tmp, _, _ = model(x_valid_seq)

    return tmp.unsqueeze(1)

In [ ]:
num_epochs = 2000
h0, c0 = None, None

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()

    inputs = x_seq.to(device)
    targets = y_seq.unsqueeze(1).to(device)

    outputs, h0, c0 = model(inputs, h0, c0)
    loss = criterion(outputs, targets)
    loss.backward()
    optimizer.step()

    h0 = h0.detach()
    c0 = c0.detach()


    if (epoch+1) % 20 == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')
        ypred = predict_on_x_valid(model, x_valid2, device)
        tmp = custom_loss(ypred, y_valid)

        print(f'Custom Loss: {tmp.item()}')




In [64]:
ypred = predict_on_x_valid(model, x_valid, device)

In [65]:
loss = custom_loss(ypred, y_valid)

print(f'Custom Loss: {loss.item()}')

Custom Loss: 1.4644279479980469


In [6]:
net = torch.load('eznnfinal.pth')

In [ ]:
meteo= pd.read_csv("./augmented_dataset.csv", sep=",")


In [8]:
target = "Mwh"
Y = prod[target]
X = meteo.drop(columns=["Dataora"])
unito = pd.concat([X, Y], axis=1)

X.head()

,wds_comuneimpianto,wdd_comuneimpianto,rnf_comuneimpianto,rhm_comuneimpianto,wds_comunelimitrofo1,wdd_comunelimitrofo1,rnf_comunelimitrofo1,rhm_comunelimitrofo1,wds_comunelimitrofo2,wdd_comunelimitrofo2,...,media_wds3_tutti,std_wds3_tutti,media_u_tutti,std_u_tutti,media_v_tutti,std_v_tutti,media_rnf_tutti,std_rnf_tutti,media_rhm_tutti,std_rhm_tutti
0,7.0,237.2,0.0,80.4,8.8,233.5,0.0,76.9,8.8,235.3,...,548.367878,159.578000,-4.782958,0.568275,-6.922163,0.766997,0.0,0.0,80.82,2.573325
1,7.0,235.8,0.0,80.0,8.9,233.1,0.0,75.7,9.1,235.4,...,597.048611,177.228827,-4.944730,0.573290,-7.076845,0.919907,0.0,0.0,80.30,2.933428
2,6.8,235.8,0.0,79.5,8.5,233.2,0.0,74.1,8.7,235.6,...,601.559789,176.713818,-4.804956,0.554989,-6.807218,0.947215,0.0,0.0,79.28,3.266803
3,6.9,236.7,0.0,78.8,8.6,231.9,0.0,73.3,8.6,234.7,...,597.938722,173.422065,-4.848669,0.607695,-6.646031,0.864030,0.0,0.0,78.66,3.357529
4,6.5,237.9,0.0,78.6,8.1,230.4,0.0,72.6,8.3,233.7,...,582.638489,182.528582,-4.769493,0.745933,-6.373543,0.929217,0.0,0.0,78.34,3.562022


In [17]:
import torch.nn as nn
prev = []

net = MLP2()
net.load_state_dict(torch.load("eznnfinal.pth"))  # load the OrderedDict into the model
net.eval()

for i in range(0,744):
    x_tensor = torch.tensor(X.iloc[i, :], dtype=torch.float32)
    prev.append(net(x_tensor))

/tmp/ipykernel_4833/622922815.py:10: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  x_tensor = torch.tensor(X.iloc[i, :], dtype=torch.float32)


In [10]:
prev

[tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.0472], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.], grad_fn=<ReluBackward0>),
 tensor([0.0345], grad_fn=<ReluBackward0>),
 tensor([0.4795], grad_fn=<ReluBackward0>),
 tensor([0.7717], grad_fn=<ReluBackward0>),
 tensor([0.7926], grad_f

In [20]:
with open('res.csv', 'w', newline='') as f:
    writer = csv.writer(f)
    for v in prev:
        writer.writerow([v.item()])  # Convert tensor to scalar value